# 04 — Exploratory Visualization

Matplotlib exploration charts with consistent display names and full race stratification.

**Labeling:** All charts EXCEPT small multiples have value labels on bars. Small multiples are exploratory only (no labels).

**Display names:** Consistent short cause labels applied across all charts via `short_name()`.

**Chart inventory:**
- Chart 1: National abortion comparison (labeled bars)
- Chart 2: Top 7 causes by race (labeled horizontal bars, 1×6 grid)
- Chart 3: Top 10 causes by sex (labeled horizontal bars, Female|Male side-by-side)
- Chart 4: Small multiples — top 10 causes × sex × age group (national) — horizontal stacked bars, NO labels
- Chart 5: Small multiples by race — top 7 causes per race × age group (6 separate 2×4 grids) — horizontal stacked bars, NO labels
- Chart 6: Top causes by race & sex (labeled, Female|Male side-by-side, 1 figure per race = 6 figures)


## Setup & Config


In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingest import load_config
from src.clean_quality import get_connection, run_sql

cfg = load_config('config.yaml')
con = get_connection(cfg)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Display name mapping for consistent labeling across all charts
DISPLAY_NAMES = {
    'Diseases of heart': 'Heart disease',
    'Malignant neoplasms': 'Cancer',
    'Chronic lower respiratory diseases': 'Respiratory disease',
    'Cerebrovascular diseases': 'Stroke',
    'Alzheimer disease': "Alzheimer's",
    'Diabetes mellitus': 'Diabetes',
    'Accidents (unintentional injuries)': 'Accidents',
    'Intentional self-harm (suicide)': 'Suicide',
    'Chronic liver disease and cirrhosis': 'Liver disease',
    'Nephritis, nephrotic syndrome and nephrosis': 'Kidney disease',
    'Influenza and pneumonia': 'Flu/Pneumonia',
    'Essential hypertension and hypertensive renal disease': 'Hypertension',
    'Assault (homicide)': 'Homicide',
    'Pregnancy, childbirth and the puerperium': 'Pregnancy/childbirth',
}

def short_name(cause):
    """Map verbose cause name to short display name."""
    return DISPLAY_NAMES.get(cause, cause)

print('✓ Setup complete')


## Load Data


In [ ]:
# National aggregated data
mort_national = run_sql('SELECT * FROM mortality_national ORDER BY deaths DESC', con)
mort_by_sex_age = run_sql('SELECT * FROM mortality_by_sex_age', con)

# Race-stratified data
mort_race_sex = run_sql('SELECT * FROM mortality_race_sex', con)
mort_race_age = run_sql('SELECT * FROM mortality_race_age', con)

print(f'✓ National data: {len(mort_national)} causes')
print(f'✓ By sex+age (national): {len(mort_by_sex_age)} rows')
print(f'✓ By race+sex: {len(mort_race_sex)} rows')
print(f'✓ By race+age (no sex): {len(mort_race_age)} rows')

# Load export
try:
    master_table = pd.read_csv('export/abortion_cause_of_death_v1.csv')
    print(f'✓ Export loaded: {len(master_table)} rows')
except FileNotFoundError:
    print('⚠ Export file not found. Run 03-prepare.ipynb or scripts/generate_export.py first.')
    master_table = None


## Chart 1: National Abortion Comparison (Labeled)


In [ ]:
if master_table is None:
    print('⚠ Skipping Chart 1: export not available')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # LEFT: Without abortion (top 5)
    without = master_table[
        (master_table['scenario'] == 'Without abortion')
    ].head(5).sort_values('deaths', ascending=True).copy()
    without['cause_display'] = without['cause'].apply(short_name)
    
    ax1.barh(without['cause_display'], without['deaths'], color='steelblue')
    ax1.set_title('Leading Causes of Death, USA 2024 (All Persons)', fontsize=13, fontweight='bold', loc='left')
    ax1.set_xlabel('')
    ax1.set_xticks([])
    for i, (cause, v) in enumerate(zip(without['cause_display'], without['deaths'])):
        ax1.text(v + 10000, i, f'{v:,.0f}', va='center', fontsize=10, fontweight='bold')
    
    # RIGHT: With abortion - show the SAME top 5 causes + abortion
    with_abort = master_table[
        (master_table['scenario'] == 'With abortion')
    ].copy()
    top_5_codes = without['cause_code'].tolist()
    same_5_in_with = with_abort[with_abort['cause_code'].isin(top_5_codes)]
    abort_only = with_abort[with_abort['cause_code'] == 'ABORT']
    with_data = pd.concat([same_5_in_with, abort_only]).sort_values('deaths', ascending=True).copy()
    with_data['cause_display'] = with_data['cause'].apply(short_name)
    
    colors = ['red' if c == 'ABORT' else 'steelblue' for c in with_data['cause_code']]
    ax2.barh(with_data['cause_display'], with_data['deaths'], color=colors)
    ax2.set_title('If Abortion Were a Cause of Death', fontsize=13, fontweight='bold', loc='left')
    ax2.set_xlabel('')
    ax2.set_xticks([])
    for i, (cause, v) in enumerate(zip(with_data['cause_display'], with_data['deaths'])):
        ax2.text(v + 10000, i, f'{v:,.0f}', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('outputs/01_national_without_vs_with.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Chart 1: National comparison saved')


## Chart 2: Top Causes by Race (Labeled Horizontal Bars)


In [ ]:
# Clean race×sex data, aggregate by race (sum across sexes)
mort_race_clean = mort_race_sex[
    (mort_race_sex['icd_10_113_cause_list'].str.startswith('#', na=False)) &
    (~mort_race_sex['single_race_6'].isin(['Not Available', '']))
].copy()

# Extract cause name from the cause_code + cause_name field
race_cause = mort_race_clean.groupby(['single_race_6', 'icd_10_113_cause_list']).agg(
    {'deaths': 'sum'}
).reset_index().sort_values(['single_race_6', 'deaths'], ascending=[True, False])

race_cause['cause'] = race_cause['icd_10_113_cause_list'].str.replace('^#\s*', '', regex=True)
race_cause['cause'] = race_cause['cause'].str.replace(r'\s*\([^)]+\)\s*$', '', regex=True)
race_cause['cause_display'] = race_cause['cause'].apply(short_name)

races = sorted(mort_race_clean['single_race_6'].unique())
print(f'Races: {races}')

fig, axes = plt.subplots(1, 6, figsize=(20, 6))
for idx, race in enumerate(races):
    ax = axes[idx]
    race_data = race_cause[
        race_cause['single_race_6'] == race
    ].head(7).sort_values('deaths', ascending=True)
    
    ax.barh(race_data['cause_display'], race_data['deaths'], color='steelblue')
    ax.set_title(race, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticks([])
    # Add value labels
    for i, (cause, v) in enumerate(zip(race_data['cause_display'], race_data['deaths'])):
        ax.text(v + 1000, i, f'{v:,.0f}', va='center', fontsize=8, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Top 7 Causes of Death by Race (2024)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/02_causes_by_race.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart 2: Top causes by race saved')


## Chart 3: Top Causes by Sex (Labeled, National)


In [ ]:
# mort_by_sex_age already has clean cause names (no # prefix)
top_by_sex = mort_by_sex_age.groupby(['sex', 'cause']).agg(
    {'deaths': 'sum'}
).reset_index().sort_values(['sex', 'deaths'], ascending=[True, False])

top_female = top_by_sex[top_by_sex['sex'] == 'Female'].head(10).copy()
top_male = top_by_sex[top_by_sex['sex'] == 'Male'].head(10).copy()

top_female['cause_display'] = top_female['cause'].apply(short_name)
top_male['cause_display'] = top_male['cause'].apply(short_name)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

top_female_sorted = top_female.sort_values('deaths', ascending=True)
ax1.barh(top_female_sorted['cause_display'], top_female_sorted['deaths'], color='indianred')
ax1.set_title('Top 10 Causes of Death — Females (National)', fontsize=13, fontweight='bold', loc='left')
ax1.set_xlabel('')
ax1.set_xticks([])
for i, (cause, v) in enumerate(zip(top_female_sorted['cause_display'], top_female_sorted['deaths'])):
    ax1.text(v + 5000, i, f'{v:,.0f}', va='center', fontsize=9, fontweight='bold')

top_male_sorted = top_male.sort_values('deaths', ascending=True)
ax2.barh(top_male_sorted['cause_display'], top_male_sorted['deaths'], color='steelblue')
ax2.set_title('Top 10 Causes of Death — Males (National)', fontsize=13, fontweight='bold', loc='left')
ax2.set_xlabel('')
ax2.set_xticks([])
for i, (cause, v) in enumerate(zip(top_male_sorted['cause_display'], top_male_sorted['deaths'])):
    ax2.text(v + 5000, i, f'{v:,.0f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/03_causes_by_sex_national.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart 3: Top causes by sex (national) saved')


## Chart 4: Small Multiples by Cause × Sex × Age (National) — NO LABELS

Horizontal stacked bars showing sex distribution by age group for each of the top 10 causes.


In [ ]:
# Get top 10 causes at national level
top_10_causes = mort_national.head(10)['cause'].tolist()

# mort_by_sex_age already has clean cause names
top_10_data = mort_by_sex_age[mort_by_sex_age['cause'].isin(top_10_causes)].copy()

age_order = [
    'Under 1 year', '1-4 years', '5-9 years', '10-14 years', '15-19 years',
    '20-24 years', '25-29 years', '30-34 years', '35-39 years', '40-44 years',
    '45-49 years', '50-54 years', '55-59 years', '60-64 years', '65-69 years',
    '70-74 years', '75-79 years', '80-84 years', '85 years and over'
]

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for idx, cause in enumerate(top_10_causes):
    ax = axes[idx]
    cause_data = top_10_data[top_10_data['cause'] == cause].copy()
    
    # Pivot: age × sex
    pivot = cause_data.pivot_table(
        index='age_group',
        columns='sex',
        values='deaths',
        aggfunc='sum',
        fill_value=0
    )
    
    # Reorder by age
    pivot = pivot.reindex([ag for ag in age_order if ag in pivot.index])
    
    # Ensure correct column order (Male, Female)
    if 'Male' in pivot.columns and 'Female' in pivot.columns:
        pivot = pivot[['Male', 'Female']]
    
    # Stacked horizontal bar chart
    pivot.plot(
        kind='barh',
        stacked=True,
        ax=ax,
        color=['steelblue', 'indianred'],
        legend=(idx == 0)
    )
    
    display_cause = short_name(cause)
    ax.set_title(display_cause, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='both', labelsize=7)
    
    if idx == 0:
        ax.legend(title='Sex', fontsize=8, title_fontsize=8, loc='lower right')
    else:
        ax.legend().remove()

plt.suptitle('Top 10 Causes of Death: Sex Distribution by Age Group — National (2024)',
             fontsize=12, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('outputs/04_causes_sex_age_multiples_national.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart 4: Small multiples (cause × sex × age, national) saved')


## Chart 5: Small Multiples by Race (Top Causes × Age) — NO LABELS

Note: `mortality_race_age` lacks sex column (aggregated), so we show age distribution only.
These are exploratory, showing top causes per race by age group.


In [ ]:
# Clean race×age data
race_age_clean = mort_race_age[
    (mort_race_age['icd_10_113_cause_list'].str.startswith('#', na=False)) &
    (~mort_race_age['single_race_6'].isin(['Not Available', '']))
].copy()

race_age_clean['cause'] = race_age_clean['icd_10_113_cause_list'].str.replace('^#\s*', '', regex=True)
race_age_clean['cause'] = race_age_clean['cause'].str.replace(r'\s*\([^)]+\)\s*$', '', regex=True)
race_age_clean['cause_display'] = race_age_clean['cause'].apply(short_name)

races_list = sorted(race_age_clean['single_race_6'].unique())
print(f'Creating small multiples for {len(races_list)} races')

for race_idx, race in enumerate(races_list):
    race_data = race_age_clean[race_age_clean['single_race_6'] == race].copy()
    
    # Get top 7 causes by total deaths in this race
    top_causes_by_race = race_data.groupby('cause')['deaths'].sum().nlargest(7).index.tolist()
    
    # Filter to top causes
    top_race_data = race_data[race_data['cause'].isin(top_causes_by_race)].copy()
    
    # Create 2×4 grid for this race (7 causes + 1 empty)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()
    
    for ax_idx, cause in enumerate(top_causes_by_race):
        ax = axes[ax_idx]
        cause_data = top_race_data[top_race_data['cause'] == cause].copy()
        
        # Pivot: age (just deaths, no sex column in race_age)
        pivot = cause_data.pivot_table(
            index='five_year_age_groups',
            values='deaths',
            aggfunc='sum'
        )
        
        # Reindex by age order (only include ages that exist in pivot)
        available_ages = [ag for ag in age_order if ag in pivot.index]
        if available_ages:
            pivot = pivot.reindex(available_ages)
        
        # Horizontal bar (no stacking, no labels)
        ax.barh(pivot.index, pivot.values.flatten(), color='steelblue')
        display_cause = short_name(cause)
        ax.set_title(display_cause, fontsize=9, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='both', labelsize=7)
        ax.set_xticks([])
    
    # Hide the last empty subplot
    axes[7].axis('off')
    
    plt.suptitle(f'Top Causes of Death by Age Group — {race} (2024)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    safe_race = race.replace('/', '_').replace(' ', '_').lower()
    plt.savefig(f'outputs/05_causes_age_multiples_race_{safe_race}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('✓ Chart 5: Small multiples by race saved (6 figures)')


## Chart 6: Top Causes by Race & Sex (Labeled, Female|Male Side-by-Side)

One figure per race showing top 7 causes split by sex.


In [ ]:
# Clean race×sex data
race_sex_clean = mort_race_sex[
    (mort_race_sex['icd_10_113_cause_list'].str.startswith('#', na=False)) &
    (~mort_race_sex['single_race_6'].isin(['Not Available', ''])) &
    (~mort_race_sex['sex'].isin(['Not Available', '']))
].copy()

race_sex_clean['cause'] = race_sex_clean['icd_10_113_cause_list'].str.replace('^#\s*', '', regex=True)
race_sex_clean['cause'] = race_sex_clean['cause'].str.replace(r'\s*\([^)]+\)\s*$', '', regex=True)
race_sex_clean['cause_display'] = race_sex_clean['cause'].apply(short_name)

races_list = sorted(race_sex_clean['single_race_6'].unique())
print(f'Creating race×sex comparisons for {len(races_list)} races')

for race in races_list:
    race_sex_data = race_sex_clean[race_sex_clean['single_race_6'] == race].copy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for ax_idx, sex in enumerate(['Female', 'Male']):
        ax = axes[ax_idx]
        sex_data = race_sex_data[
            race_sex_data['sex'] == sex
        ].nlargest(7, 'deaths').sort_values('deaths', ascending=True)
        
        color = 'indianred' if sex == 'Female' else 'steelblue'
        ax.barh(sex_data['cause_display'], sex_data['deaths'], color=color)
        ax.set_title(f'{sex}', fontsize=12, fontweight='bold')
        ax.set_xlabel('')
        ax.set_xticks([])
        # Add value labels
        for i, (cause, v) in enumerate(zip(sex_data['cause_display'], sex_data['deaths'])):
            ax.text(v + 500, i, f'{v:,.0f}', va='center', fontsize=9, fontweight='bold')
        ax.tick_params(axis='y', labelsize=9)
    
    safe_race = race.replace('/', '_').replace(' ', '_').lower()
    plt.suptitle(f'Top Causes of Death by Sex — {race} (2024)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'outputs/06_causes_by_sex_race_{safe_race}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('✓ Chart 6: Top causes by race & sex saved (6 figures)')


## Cleanup


In [ ]:
con.close()
print('\n✓✓✓ All visualizations complete')
print('\nGenerated:' )
print('  Chart 1: 01_national_without_vs_with.png')
print('  Chart 2: 02_causes_by_race.png')
print('  Chart 3: 03_causes_by_sex_national.png')
print('  Chart 4: 04_causes_sex_age_multiples_national.png')
print('  Chart 5: 05_causes_age_multiples_race_*.png (6 figures, 1 per race)')
print('  Chart 6: 06_causes_by_sex_race_*.png (6 figures, 1 per race)')
